# 02 — Ablation Study

Sweep chunk-size {200,400,800,1600} и vector vs hybrid. Метрика recall@5 на 16 вопросах.

> Офлайн/CI — hash-эмбеддер. Для настоящих цифр: `EMBEDDING_BACKEND=st` + реальный корпус.

In [ ]:
import os; os.environ.setdefault('EMBEDDING_BACKEND','hash')
import sys; sys.path.insert(0,'..'); sys.path.insert(0,'../evaluation')
sys.path.insert(0, str(ROOT / 'scripts'))
from build_index import build_chunks
from app.retrieval import Retriever, FaissStore, build_embedder
from evaluate_retrieval import evaluate, load_gt
GT=load_gt(); RAW='../data/raw'; emb=build_embedder()

## 1. Chunk-size sweep (recursive, hybrid)

In [ ]:
rows=[]
for size in [200,400,800,1600]:
    ch=build_chunks(RAW,'recursive',size,int(size*0.15))
    st=FaissStore.build(emb.encode([c.text for c in ch]),ch,emb.model_name,'v1')
    m=evaluate(Retriever(emb,st,'hybrid'),GT,5)
    rows.append((size,len(ch),m['recall@5'],m['MRR']))
    print(f"size={size:>4} chunks={len(ch):>3} recall@5={m['recall@5']:.3f} MRR={m['MRR']:.3f}")

## 2. Section + vector vs hybrid

In [ ]:
ch=build_chunks(RAW,'section')
st=FaissStore.build(emb.encode([c.text for c in ch]),ch,emb.model_name,'v1')
for mode in ['vector','hybrid']:
    m=evaluate(Retriever(emb,st,mode),GT,5)
    print(f"{mode:>7}: recall@5={m['recall@5']:.3f} MRR={m['MRR']:.3f} nDCG@10={m['nDCG@10']:.3f}")

## 3. Plot -> results/ (для README §6)

In [ ]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
plt.figure(figsize=(6,4)); plt.plot([r[0] for r in rows],[r[2] for r in rows],marker='o')
plt.axhline(0.75,ls='--',color='r',label='порог 0.75')
plt.xlabel('chunk size'); plt.ylabel('recall@5'); plt.title('Chunk-size ablation'); plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig('../evaluation/results/ablation_chunksize.png',dpi=120); print('saved')

## 4. Вывод (заполнить по своим числам)

> Пример: «section даёт recall@5=X.XX против Y.YY у recursive-800, потому что статьи самодостаточны; hybrid добавляет +Z за счёт точного лексического матча номеров статей.»